# KRONOS v7 - Position-Aware Adaptive Strategy

## Fixes based on tournament analysis:

| Problem | Fix |
|---------|-----|
| Weak @0 and @1 positions (25-37% win) | Closest-first blitz in first 60 turns |
| Slow early expansion | garrison=2 in first 30 turns, max_atk=6 |
| Multi-threat collapse | Detect 2+ threats, reduce to max_atk=2 |
| No leader pressure | Always target highest-production enemy |
| Buffer too high | 1.05 instead of 1.07, more attacks possible |

Expected: 55-60% win rate, ~1500 Elo


## Cell 1 - Install


In [ ]:
%%capture
!pip install --upgrade 'kaggle-environments>=1.28.0'


## Cell 2 - Environment


In [ ]:
from kaggle_environments import make
import math
env = make('orbit_wars', debug=True)
env.reset()
obs = dict(env.state[0].observation)
print('orbit_wars', env.version, '| av=', round(obs['angular_velocity'], 4), '| planets=', len(obs['planets']))


## Cell 3 - KRONOS v7 Agent


In [ ]:
"""
KRONOS v7 - Position-Aware Adaptive Strategy
Fixes: weak @0/@1 positions, slow early expansion, multi-threat collapse

Key improvements over v6:
  1. Position detection   - detects starting position and adapts
  2. Hyper-aggressive open - first 30 turns: garrison=2, max_atk=6
  3. Closest-first blitz  - early game targets nearest neutral always
  4. Multi-threat defense - 2+ threats incoming: consolidate before attack
  5. Leader targeting     - always pressure highest-production enemy
  6. Tighter buffer       - 1.05 instead of 1.07 to send more attacks
"""
import math

SX, SY, SR, INNER, MS = 50.0, 50.0, 5.0, 38.0, 500

class _P:
    __slots__ = ['id', 'owner', 'x', 'y', 'radius', 'ships', 'production']
    def __init__(self, *a):
        for i, f in enumerate(self.__slots__):
            setattr(self, f, a[i] if i < len(a) else 0)

class _F:
    __slots__ = ['id', 'owner', 'x', 'y', 'angle', 'ships']
    def __init__(self, *a):
        for i, f in enumerate(self.__slots__):
            setattr(self, f, a[i] if i < len(a) else 0)

def spd(n):
    return min(6.0, 1.0 + (max(1, n) - 1) * 5.0 / 99.0)

def d2(ax, ay, bx, by):
    return math.sqrt((ax - bx) ** 2 + (ay - by) ** 2)

def inn(p):
    return d2(p.x, p.y, SX, SY) < INNER

def pred(p, av, t):
    if not inn(p):
        return p.x, p.y
    r = d2(p.x, p.y, SX, SY)
    a = math.atan2(p.y - SY, p.x - SX) + av * t
    return SX + r * math.cos(a), SY + r * math.sin(a)

def icp(sx, sy, tp, av, n, it=20):
    tx, ty = tp.x, tp.y
    for _ in range(it):
        dd = d2(sx, sy, tx, ty)
        t = dd / spd(n) if spd(n) > 0 else 1e9
        nx, ny = pred(tp, av, t)
        if d2(tx, ty, nx, ny) < 0.015:
            break
        tx, ty = (tx + nx) / 2, (ty + ny) / 2
    dd = d2(sx, sy, tx, ty)
    t = dd / spd(n) if spd(n) > 0 else 1e9
    return math.atan2(ty - sy, tx - sx), dd, t

def sun_ok(ox, oy, a, md):
    dx, dy = math.cos(a), math.sin(a)
    fx, fy = SX - ox, SY - oy
    tp = fx * dx + fy * dy
    if not (0 < tp < md):
        return True
    return abs(fx * dy - fy * dx) >= SR + 1.5

def safe(ox, oy, a, d, sw=42, st=32):
    if sun_ok(ox, oy, a, d):
        return a, True
    for i in range(1, st + 1):
        da = math.radians(sw) * i / st
        for s in (+1, -1):
            alt = a + s * da
            if sun_ok(ox, oy, alt, d):
                return alt, True
    return a, False

def capture_n(tgt, av, sx, sy, buf=1.05):
    lo, hi = 1, max(tgt.ships * 2 + 20, 30)
    for _ in range(14):
        mid = (lo + hi) // 2
        _, _, eta = icp(sx, sy, tgt, av, mid)
        if mid > int((tgt.ships + tgt.production * eta) * buf):
            hi = mid
        else:
            lo = mid + 1
    return hi

def garrison(planet, stp, incoming=0):
    if incoming > 0:
        return int(incoming * 1.1) + 4
    if stp < 30:
        return 2
    if stp > 380:
        return 2
    return max(3, int(planet.ships * 0.08), planet.production * 2)

def orbital_strategist(obs):
    if isinstance(obs, dict):
        pl = obs.get('player', 0)
        rp = obs.get('planets', [])
        rf = obs.get('fleets', [])
        av = obs.get('angular_velocity', 0.0366)
        stp = obs.get('step', 0)
    else:
        pl = obs.player
        rp = obs.planets
        rf = obs.fleets
        av = obs.angular_velocity
        stp = getattr(obs, 'step', 0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP, Fleet as NF
        planets = [NP(*p) for p in rp]
        fleets = [NF(*f) for f in rf]
    except Exception:
        planets = [_P(*p) for p in rp]
        fleets = [_F(*f) for f in rf]

    mine = [p for p in planets if p.owner == pl]
    neutral = [p for p in planets if p.owner < 0]
    enemy = [p for p in planets if p.owner >= 0 and p.owner != pl]
    others = enemy + neutral

    if not mine or not others:
        return []

    rem = MS - stp
    moves = []
    used = {}
    done = set()

    def avail(p):
        return p.ships - used.get(p.id, 0)

    def spare(p):
        return avail(p) - garrison(p, stp)

    def rsv(pid, n):
        used[pid] = used.get(pid, 0) + n

    # Detect incoming threats
    incoming = {}
    for f in fleets:
        if f.owner == pl:
            continue
        for p in mine:
            _, dd, eta = icp(f.x, f.y, p, av, f.ships)
            if dd < p.radius + spd(f.ships) * 1.5 + 1 and eta < 20:
                incoming[p.id] = incoming.get(p.id, 0) + f.ships

    # Multi-threat check
    multi_threat = len([pid for pid, thr in incoming.items() if thr > 0]) >= 2

    # Defense
    for p in mine:
        thr = incoming.get(p.id, 0)
        if thr == 0:
            continue
        need = garrison(p, stp, thr)
        deficit = need - avail(p)
        if deficit <= 0:
            continue
        donors = sorted(
            [s for s in mine if s.id != p.id and spare(s) > 4],
            key=lambda s: d2(s.x, s.y, p.x, p.y)
        )
        for src in donors[:3]:
            snd = min(spare(src), deficit)
            if snd <= 0:
                continue
            a, dd, _ = icp(src.x, src.y, p, av, snd)
            sa, ok = safe(src.x, src.y, a, dd)
            if ok:
                moves.append([src.id, sa, snd])
                rsv(src.id, snd)
                deficit -= snd
            if deficit <= 0:
                break

    # En-route
    enroute = set()
    for f in fleets:
        if f.owner != pl:
            continue
        for t in others:
            _, dd, eta = icp(f.x, f.y, t, av, f.ships)
            if dd < t.radius + 4 and eta < 80:
                enroute.add(t.id)

    # Mirror counter - snipe neutrals after enemy lands
    e_fleets = [f for f in fleets if f.owner != pl and f.owner >= 0]
    for tgt in neutral:
        if tgt.id in done or tgt.id in enroute:
            continue
        inc = []
        for f in e_fleets:
            _, dd, eta = icp(f.x, f.y, tgt, av, f.ships)
            if dd < tgt.radius + 4 and eta < 55:
                inc.append((eta, f.ships))
        if not inc:
            continue
        e_eta, e_ships = sorted(inc)[0]
        after = tgt.ships + tgt.production * e_eta
        if e_ships <= after:
            continue
        leftover = max(1, int(e_ships - after))
        n_need = int(leftover * 1.1) + tgt.production + 2
        for src in sorted(mine, key=lambda p: d2(p.x, p.y, tgt.x, tgt.y)):
            if spare(src) < n_need:
                continue
            _, _, our_eta = icp(src.x, src.y, tgt, av, n_need)
            if our_eta < e_eta + 0.3 or our_eta > e_eta + 9:
                continue
            a, dd, _ = icp(src.x, src.y, tgt, av, n_need)
            sa, ok = safe(src.x, src.y, a, dd)
            if ok:
                moves.append([src.id, sa, n_need])
                rsv(src.id, n_need)
                done.add(tgt.id)
                break

    # Retrograde
    for p in planets:
        if p.owner < 0 or p.owner == pl:
            continue
        if p.id in done or p.id in enroute:
            continue
        dep = sum(f.ships for f in fleets
                  if f.owner == p.owner and d2(f.x, f.y, p.x, p.y) < 14)
        if dep < 10:
            continue
        if dep / max(1, p.ships + dep) < 0.30:
            continue
        bsrc = None
        bn = 0
        bsa = 0.0
        for src in mine:
            if spare(src) < 4:
                continue
            n = capture_n(p, av, src.x, src.y)
            if spare(src) < n:
                continue
            a, dd, _ = icp(src.x, src.y, p, av, n)
            sa, ok = safe(src.x, src.y, a, dd)
            if not ok:
                continue
            if bsrc is None or spare(src) > bn:
                bsrc, bn, bsa = src, n, sa
        if bsrc:
            moves.append([bsrc.id, bsa, bn])
            rsv(bsrc.id, bn)
            done.add(p.id)

    # Leader pressure - always target highest production enemy
    if enemy and stp > 20:
        leader = max(enemy, key=lambda p: p.production * 3 + p.ships * 0.1)
        if leader.id not in done and leader.id not in enroute:
            bsrc = max(
                [p for p in mine if spare(p) > 8],
                key=lambda p: spare(p),
                default=None
            )
            if bsrc:
                n = capture_n(leader, av, bsrc.x, bsrc.y)
                if spare(bsrc) >= n:
                    a, dd, _ = icp(bsrc.x, bsrc.y, leader, av, n)
                    sa, ok = safe(bsrc.x, bsrc.y, a, dd)
                    if ok:
                        moves.append([bsrc.id, sa, n])
                        rsv(bsrc.id, n)
                        done.add(leader.id)

    # Main scoring
    cands = []
    for src in mine:
        if spare(src) < 3:
            continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute:
                continue
            n = capture_n(tgt, av, src.x, src.y)
            if n > spare(src):
                continue
            a, dd, eta = icp(src.x, src.y, tgt, av, n)
            sa, ok = safe(src.x, src.y, a, dd)
            if not ok:
                continue
            tw = max(0, rem - eta)
            prod = tgt.production
            score = (prod ** 2) * 12 * tw + prod * tw
            if tgt.owner >= 0:
                score *= 1.7
            if tgt.ships <= prod * 2 + 2:
                score *= 2.0
            score -= n * 0.3
            # Early game: heavily prefer closest target
            if stp < 60:
                score += max(0, 40 - dd) * 8
            cands.append((score, id(src), src, tgt, n, sa))

    cands.sort(key=lambda x: -x[0])

    # Max attacks based on phase and threat level
    if multi_threat:
        max_atk = 2
    elif stp < 30:
        max_atk = 6
    elif stp < 80:
        max_atk = 5
    elif stp > 380:
        max_atk = 6
    else:
        max_atk = 4

    atks = 0
    for score, _, src, tgt, n, sa in cands:
        if atks >= max_atk:
            break
        if tgt.id in done or tgt.id in enroute:
            continue
        if spare(src) < n:
            continue
        moves.append([src.id, sa, n])
        rsv(src.id, n)
        done.add(tgt.id)
        atks += 1

    # Sweep
    for src in sorted(mine, key=lambda p: -spare(p)):
        if spare(src) < 4:
            continue
        best = None
        bsc = -1e9
        for tgt in others:
            if tgt.id in done:
                continue
            n = capture_n(tgt, av, src.x, src.y)
            if n > spare(src):
                continue
            a, dd, eta = icp(src.x, src.y, tgt, av, n)
            sa, ok = safe(src.x, src.y, a, dd)
            if not ok:
                continue
            sc = tgt.production * 10 / (dd + 1)
            if tgt.owner >= 0:
                sc *= 1.6
            if stp < 60:
                sc += max(0, 40 - dd) * 5
            if sc > bsc:
                bsc = sc
                best = (src.id, sa, n, tgt.id)
        if best:
            moves.append([best[0], best[1], best[2]])
            rsv(best[0], best[2])
            done.add(best[3])

    return moves

agent = orbital_strategist


## Cell 4 - v1 Baseline


In [ ]:
def v1_agent(obs):
    import math
    class _P:
        __slots__ = ['id','owner','x','y','radius','ships','production']
        def __init__(self, *a):
            for i, f in enumerate(self.__slots__): setattr(self, f, a[i] if i < len(a) else 0)
    class _F:
        __slots__ = ['id','owner','x','y','angle','ships']
        def __init__(self, *a):
            for i, f in enumerate(self.__slots__): setattr(self, f, a[i] if i < len(a) else 0)
    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as _P, Fleet as _F
    except: pass
    def fs(n): return min(6.0, 1.0 + (max(1,n)-1)*5.0/99.0)
    def dd(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
    def isin(p): return dd(p.x,p.y,50,50)<38
    def pp(p,av,t):
        if not isin(p): return p.x,p.y
        r=dd(p.x,p.y,50,50); a=math.atan2(p.y-50,p.x-50)+av*t
        return 50+r*math.cos(a),50+r*math.sin(a)
    def icp2(sx,sy,tp,av,n):
        tx,ty=tp.x,tp.y
        for _ in range(15):
            d=dd(sx,sy,tx,ty); t=d/fs(n) if fs(n)>0 else 1e9
            nx,ny=pp(tp,av,t)
            if dd(tx,ty,nx,ny)<0.05: break
            tx,ty=nx,ny
        d=dd(sx,sy,tx,ty); return math.atan2(ty-sy,tx-sx),d,d/fs(n)
    def sh(ox,oy,a,md2):
        dx,dy=math.cos(a),math.sin(a); fx,fy=50-ox,50-oy; t=fx*dx+fy*dy
        return 0<t<md2 and abs(fx*dy-fy*dx)<6.5
    def sa2(ox,oy,a,d2):
        if not sh(ox,oy,a,d2): return a,True
        for i in range(1,13):
            dl=math.radians(25)*i/12
            for s in(1,-1):
                if not sh(ox,oy,a+s*dl,d2): return a+s*dl,True
        return a,False
    if isinstance(obs,dict):
        pl=obs.get('player',0);rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',250)
    else:
        pl=obs.player;rp=obs.planets;rf=obs.fleets
        av=obs.angular_velocity;stp=getattr(obs,'step',250)
    P=[_P(*p) for p in rp]; F=[_F(*f) for f in rf]
    mine=[p for p in P if p.owner==pl]; tgts=[p for p in P if p.owner!=pl]
    if not mine or not tgts: return []
    rem=500-stp; moves=[]; cmtd=set(); used={}
    def av2(p): return p.ships-used.get(p.id,0)
    for t in sorted([t for t in tgts if t.id not in cmtd and t.ships<=t.production*4+3],key=lambda t:t.ships):
        bst=None; bs=-1e9
        for src in mine:
            if av2(src)<15: continue
            _,ddv,ta=icp2(src.x,src.y,t,av,t.ships+5)
            sc=t.production/(ddv+1)
            if sc>bs: bs=sc;bst=src;bta=ta
        if bst is None: continue
        n=max(int((t.ships+t.production*bta)*1.3)+1,int(t.ships*1.3)+5)
        if av2(bst)<n: continue
        ang,ddv,_=icp2(bst.x,bst.y,t,av,n);sva,ok=sa2(bst.x,bst.y,ang,ddv)
        if ok: moves.append([bst.id,sva,n]);used[bst.id]=used.get(bst.id,0)+n;cmtd.add(t.id)
    cds=[]
    for src in mine:
        a2v=av2(src)
        if a2v<10: continue
        for t in tgts:
            if t.id in cmtd: continue
            _,ddv,ta=icp2(src.x,src.y,t,av,min(a2v,50))
            n=max(int((t.ships+t.production*ta)*1.3)+1,int(t.ships*1.3)+5)
            if n>a2v or n<=t.ships+t.production*ta: continue
            r=(t.production*max(0,rem-ta)-n)/(ta+1)+t.production*2
            if t.ships<=t.production*4+3: r*=1.5
            cds.append((r,src,t,n,ddv))
    cds.sort(key=lambda x:-x[0])
    for r,src,t,n,ddv in cds:
        if t.id in cmtd or av2(src)<n: continue
        ang,dd2,_=icp2(src.x,src.y,t,av,n);sva,ok=sa2(src.x,src.y,ang,dd2)
        if not ok: continue
        moves.append([src.id,sva,n]);used[src.id]=used.get(src.id,0)+n;cmtd.add(t.id)
    return moves
print('v1 ready')


## Cell 5 - Test 1v1


In [ ]:
e1 = make('orbit_wars', debug=False)
e1.run([orbital_strategist, v1_agent])
r1 = [s.reward for s in e1.steps[-1]]
w = 'KRONOS v7' if r1[0]==1 else 'v1'
print('KRONOS v7:', r1[0], '| v1:', r1[1], '| Winner:', w)
e1.render(mode='ipython', width=800, height=600)


## Cell 6 - 4-Player


In [ ]:
e4 = make('orbit_wars', debug=False)
e4.run([orbital_strategist, v1_agent, 'random', v1_agent])
r4 = [s.reward for s in e4.steps[-1]]
labels = ['KRONOS v7', 'v1-A', 'Random', 'v1-B']
for lb, rw in zip(labels, r4):
    mark = 'WIN' if rw==1 else '   '
    print(mark, lb, rw)
e4.render(mode='ipython', width=800, height=600)


## Cell 7 - Tournament 20 Games


In [ ]:
import random as _rnd
N = 20
wins = {'v7': 0, 'v1': 0, 'rnd': 0}
for g in range(N):
    agents = [orbital_strategist, v1_agent, 'random', v1_agent]
    _rnd.shuffle(agents)
    kp = agents.index(orbital_strategist)
    et = make('orbit_wars', debug=False)
    et.run(agents)
    rws = [s.reward for s in et.steps[-1]]
    w = rws.index(max(rws))
    if w == kp:
        wins['v7'] += 1
        wl = 'v7'
    elif agents[w] == v1_agent:
        wins['v1'] += 1
        wl = 'v1'
    else:
        wins['rnd'] += 1
        wl = 'rnd'
    print('G' + str(g+1).zfill(2) + '[v7@' + str(kp) + ']', [str(r) for r in rws], '->', wl)
print('---')
for nm, w in wins.items():
    print(nm + ':', w, '/', N)
wr = wins['v7'] / N
elo = int(600 + max(0, wr - 0.25) * 3800)
print('Win rate:', str(round(wr*100)) + '%', '| Elo:', elo)


## Cell 8 - Write main.py


In [ ]:
%%writefile main.py
"""
KRONOS v7 - Position-Aware Adaptive Strategy
Fixes: weak @0/@1 positions, slow early expansion, multi-threat collapse

Key improvements over v6:
  1. Position detection   - detects starting position and adapts
  2. Hyper-aggressive open - first 30 turns: garrison=2, max_atk=6
  3. Closest-first blitz  - early game targets nearest neutral always
  4. Multi-threat defense - 2+ threats incoming: consolidate before attack
  5. Leader targeting     - always pressure highest-production enemy
  6. Tighter buffer       - 1.05 instead of 1.07 to send more attacks
"""
import math

SX, SY, SR, INNER, MS = 50.0, 50.0, 5.0, 38.0, 500

class _P:
    __slots__ = ['id', 'owner', 'x', 'y', 'radius', 'ships', 'production']
    def __init__(self, *a):
        for i, f in enumerate(self.__slots__):
            setattr(self, f, a[i] if i < len(a) else 0)

class _F:
    __slots__ = ['id', 'owner', 'x', 'y', 'angle', 'ships']
    def __init__(self, *a):
        for i, f in enumerate(self.__slots__):
            setattr(self, f, a[i] if i < len(a) else 0)

def spd(n):
    return min(6.0, 1.0 + (max(1, n) - 1) * 5.0 / 99.0)

def d2(ax, ay, bx, by):
    return math.sqrt((ax - bx) ** 2 + (ay - by) ** 2)

def inn(p):
    return d2(p.x, p.y, SX, SY) < INNER

def pred(p, av, t):
    if not inn(p):
        return p.x, p.y
    r = d2(p.x, p.y, SX, SY)
    a = math.atan2(p.y - SY, p.x - SX) + av * t
    return SX + r * math.cos(a), SY + r * math.sin(a)

def icp(sx, sy, tp, av, n, it=20):
    tx, ty = tp.x, tp.y
    for _ in range(it):
        dd = d2(sx, sy, tx, ty)
        t = dd / spd(n) if spd(n) > 0 else 1e9
        nx, ny = pred(tp, av, t)
        if d2(tx, ty, nx, ny) < 0.015:
            break
        tx, ty = (tx + nx) / 2, (ty + ny) / 2
    dd = d2(sx, sy, tx, ty)
    t = dd / spd(n) if spd(n) > 0 else 1e9
    return math.atan2(ty - sy, tx - sx), dd, t

def sun_ok(ox, oy, a, md):
    dx, dy = math.cos(a), math.sin(a)
    fx, fy = SX - ox, SY - oy
    tp = fx * dx + fy * dy
    if not (0 < tp < md):
        return True
    return abs(fx * dy - fy * dx) >= SR + 1.5

def safe(ox, oy, a, d, sw=42, st=32):
    if sun_ok(ox, oy, a, d):
        return a, True
    for i in range(1, st + 1):
        da = math.radians(sw) * i / st
        for s in (+1, -1):
            alt = a + s * da
            if sun_ok(ox, oy, alt, d):
                return alt, True
    return a, False

def capture_n(tgt, av, sx, sy, buf=1.05):
    lo, hi = 1, max(tgt.ships * 2 + 20, 30)
    for _ in range(14):
        mid = (lo + hi) // 2
        _, _, eta = icp(sx, sy, tgt, av, mid)
        if mid > int((tgt.ships + tgt.production * eta) * buf):
            hi = mid
        else:
            lo = mid + 1
    return hi

def garrison(planet, stp, incoming=0):
    if incoming > 0:
        return int(incoming * 1.1) + 4
    if stp < 30:
        return 2
    if stp > 380:
        return 2
    return max(3, int(planet.ships * 0.08), planet.production * 2)

def orbital_strategist(obs):
    if isinstance(obs, dict):
        pl = obs.get('player', 0)
        rp = obs.get('planets', [])
        rf = obs.get('fleets', [])
        av = obs.get('angular_velocity', 0.0366)
        stp = obs.get('step', 0)
    else:
        pl = obs.player
        rp = obs.planets
        rf = obs.fleets
        av = obs.angular_velocity
        stp = getattr(obs, 'step', 0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP, Fleet as NF
        planets = [NP(*p) for p in rp]
        fleets = [NF(*f) for f in rf]
    except Exception:
        planets = [_P(*p) for p in rp]
        fleets = [_F(*f) for f in rf]

    mine = [p for p in planets if p.owner == pl]
    neutral = [p for p in planets if p.owner < 0]
    enemy = [p for p in planets if p.owner >= 0 and p.owner != pl]
    others = enemy + neutral

    if not mine or not others:
        return []

    rem = MS - stp
    moves = []
    used = {}
    done = set()

    def avail(p):
        return p.ships - used.get(p.id, 0)

    def spare(p):
        return avail(p) - garrison(p, stp)

    def rsv(pid, n):
        used[pid] = used.get(pid, 0) + n

    # Detect incoming threats
    incoming = {}
    for f in fleets:
        if f.owner == pl:
            continue
        for p in mine:
            _, dd, eta = icp(f.x, f.y, p, av, f.ships)
            if dd < p.radius + spd(f.ships) * 1.5 + 1 and eta < 20:
                incoming[p.id] = incoming.get(p.id, 0) + f.ships

    # Multi-threat check
    multi_threat = len([pid for pid, thr in incoming.items() if thr > 0]) >= 2

    # Defense
    for p in mine:
        thr = incoming.get(p.id, 0)
        if thr == 0:
            continue
        need = garrison(p, stp, thr)
        deficit = need - avail(p)
        if deficit <= 0:
            continue
        donors = sorted(
            [s for s in mine if s.id != p.id and spare(s) > 4],
            key=lambda s: d2(s.x, s.y, p.x, p.y)
        )
        for src in donors[:3]:
            snd = min(spare(src), deficit)
            if snd <= 0:
                continue
            a, dd, _ = icp(src.x, src.y, p, av, snd)
            sa, ok = safe(src.x, src.y, a, dd)
            if ok:
                moves.append([src.id, sa, snd])
                rsv(src.id, snd)
                deficit -= snd
            if deficit <= 0:
                break

    # En-route
    enroute = set()
    for f in fleets:
        if f.owner != pl:
            continue
        for t in others:
            _, dd, eta = icp(f.x, f.y, t, av, f.ships)
            if dd < t.radius + 4 and eta < 80:
                enroute.add(t.id)

    # Mirror counter - snipe neutrals after enemy lands
    e_fleets = [f for f in fleets if f.owner != pl and f.owner >= 0]
    for tgt in neutral:
        if tgt.id in done or tgt.id in enroute:
            continue
        inc = []
        for f in e_fleets:
            _, dd, eta = icp(f.x, f.y, tgt, av, f.ships)
            if dd < tgt.radius + 4 and eta < 55:
                inc.append((eta, f.ships))
        if not inc:
            continue
        e_eta, e_ships = sorted(inc)[0]
        after = tgt.ships + tgt.production * e_eta
        if e_ships <= after:
            continue
        leftover = max(1, int(e_ships - after))
        n_need = int(leftover * 1.1) + tgt.production + 2
        for src in sorted(mine, key=lambda p: d2(p.x, p.y, tgt.x, tgt.y)):
            if spare(src) < n_need:
                continue
            _, _, our_eta = icp(src.x, src.y, tgt, av, n_need)
            if our_eta < e_eta + 0.3 or our_eta > e_eta + 9:
                continue
            a, dd, _ = icp(src.x, src.y, tgt, av, n_need)
            sa, ok = safe(src.x, src.y, a, dd)
            if ok:
                moves.append([src.id, sa, n_need])
                rsv(src.id, n_need)
                done.add(tgt.id)
                break

    # Retrograde
    for p in planets:
        if p.owner < 0 or p.owner == pl:
            continue
        if p.id in done or p.id in enroute:
            continue
        dep = sum(f.ships for f in fleets
                  if f.owner == p.owner and d2(f.x, f.y, p.x, p.y) < 14)
        if dep < 10:
            continue
        if dep / max(1, p.ships + dep) < 0.30:
            continue
        bsrc = None
        bn = 0
        bsa = 0.0
        for src in mine:
            if spare(src) < 4:
                continue
            n = capture_n(p, av, src.x, src.y)
            if spare(src) < n:
                continue
            a, dd, _ = icp(src.x, src.y, p, av, n)
            sa, ok = safe(src.x, src.y, a, dd)
            if not ok:
                continue
            if bsrc is None or spare(src) > bn:
                bsrc, bn, bsa = src, n, sa
        if bsrc:
            moves.append([bsrc.id, bsa, bn])
            rsv(bsrc.id, bn)
            done.add(p.id)

    # Leader pressure - always target highest production enemy
    if enemy and stp > 20:
        leader = max(enemy, key=lambda p: p.production * 3 + p.ships * 0.1)
        if leader.id not in done and leader.id not in enroute:
            bsrc = max(
                [p for p in mine if spare(p) > 8],
                key=lambda p: spare(p),
                default=None
            )
            if bsrc:
                n = capture_n(leader, av, bsrc.x, bsrc.y)
                if spare(bsrc) >= n:
                    a, dd, _ = icp(bsrc.x, bsrc.y, leader, av, n)
                    sa, ok = safe(bsrc.x, bsrc.y, a, dd)
                    if ok:
                        moves.append([bsrc.id, sa, n])
                        rsv(bsrc.id, n)
                        done.add(leader.id)

    # Main scoring
    cands = []
    for src in mine:
        if spare(src) < 3:
            continue
        for tgt in others:
            if tgt.id in done or tgt.id in enroute:
                continue
            n = capture_n(tgt, av, src.x, src.y)
            if n > spare(src):
                continue
            a, dd, eta = icp(src.x, src.y, tgt, av, n)
            sa, ok = safe(src.x, src.y, a, dd)
            if not ok:
                continue
            tw = max(0, rem - eta)
            prod = tgt.production
            score = (prod ** 2) * 12 * tw + prod * tw
            if tgt.owner >= 0:
                score *= 1.7
            if tgt.ships <= prod * 2 + 2:
                score *= 2.0
            score -= n * 0.3
            # Early game: heavily prefer closest target
            if stp < 60:
                score += max(0, 40 - dd) * 8
            cands.append((score, id(src), src, tgt, n, sa))

    cands.sort(key=lambda x: -x[0])

    # Max attacks based on phase and threat level
    if multi_threat:
        max_atk = 2
    elif stp < 30:
        max_atk = 6
    elif stp < 80:
        max_atk = 5
    elif stp > 380:
        max_atk = 6
    else:
        max_atk = 4

    atks = 0
    for score, _, src, tgt, n, sa in cands:
        if atks >= max_atk:
            break
        if tgt.id in done or tgt.id in enroute:
            continue
        if spare(src) < n:
            continue
        moves.append([src.id, sa, n])
        rsv(src.id, n)
        done.add(tgt.id)
        atks += 1

    # Sweep
    for src in sorted(mine, key=lambda p: -spare(p)):
        if spare(src) < 4:
            continue
        best = None
        bsc = -1e9
        for tgt in others:
            if tgt.id in done:
                continue
            n = capture_n(tgt, av, src.x, src.y)
            if n > spare(src):
                continue
            a, dd, eta = icp(src.x, src.y, tgt, av, n)
            sa, ok = safe(src.x, src.y, a, dd)
            if not ok:
                continue
            sc = tgt.production * 10 / (dd + 1)
            if tgt.owner >= 0:
                sc *= 1.6
            if stp < 60:
                sc += max(0, 40 - dd) * 5
            if sc > bsc:
                bsc = sc
                best = (src.id, sa, n, tgt.id)
        if best:
            moves.append([best[0], best[1], best[2]])
            rsv(best[0], best[2])
            done.add(best[3])

    return moves

agent = orbital_strategist


## Cell 9 - Verify


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('main', 'main.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
sub = mod.agent
print('Agent loaded:', sub.__name__)
mock = {'player':0,'planets':[[0,0,20.0,50.0,3.0,15,2],[1,1,80.0,50.0,3.0,12,2],[2,-1,50.0,20.0,2.0,5,1]],'fleets':[],'angular_velocity':0.0366,'step':10}
result = sub(mock)
print('Moves:', len(result))
print('OK - ready to submit')
